# Pooled-path test comparison

Does not retrain. Builds one test table (and feature ranks) for the plots and the paper.

Pooled-path MAE/RMSE/R² for horizon *H* is the error on **all H lead days** stacked, in µg/m³. Lead 1 is next-day only.

Climatology is added later by `climatology_baseline.ipynb`.


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HERE = Path(".")
CNN_CSV = HERE / "cnn_lstm_results" / "cnn_lstm_test_metrics.csv"
TRANS_CSV = HERE / "transformer_results" / "transformer_test_metrics.csv"
RF_CSV = HERE / "rf_results_path" / "random_forest_pooled_path_results.csv"
XGB_CSV = HERE / "xgb_results_path" / "xgboost_pooled_path_results.csv"
RF_PRED = HERE / "rf_results_path"
XGB_PRED = HERE / "xgb_results_path"
LSTM_GRU_CSV = HERE / "true_multi_horizon_results" / "multi_horizon_metrics_original_scale.csv"
RANK_CSV = HERE / "pooled_path_feature_importance_ranks.csv"

METRIC_COLS = ["MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"]


def regression_metrics(y_true, y_pred):
    """MAE, RMSE, R², MAPE (%), MSE, MBE on flattened arrays (µg/m³)."""
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100),
        "MBE": float(np.mean(y_pred - y_true)),
    }


def labelled(df, model):
    out = df.copy()
    out.insert(0, "Model", model)  # Model column at the front, same name for every row
    return out


## Load model scores

CNN-LSTM / Transformer / LSTM / GRU: existing test CSVs.
RF / XGBoost: pooled-path 7/14/30 plus the 1-day specialist.
Persistence: PM2.5 at origin *t* held flat (same origins as the trees).


In [11]:
cnn = labelled(pd.read_csv(CNN_CSV), "CNN-LSTM")
transformer = labelled(pd.read_csv(TRANS_CSV), "Transformer")

mh = pd.read_csv(LSTM_GRU_CSV).rename(
    columns={"Horizon": "Horizon_Days", "MAPE (%)": "MAPE", "R²": "R2"}
)
mh_cols = ["Horizon_Days", "Model", *METRIC_COLS]  # * unpacks METRIC_COLS into this list
lstm_cmp = mh.loc[mh["Model"] == "LSTM", mh_cols].copy()
gru_cmp = mh.loc[mh["Model"] == "GRU", mh_cols].copy()

rf = pd.read_csv(RF_CSV) # Read Random Forest scores
xgb = pd.read_csv(XGB_CSV) # Read XGBoost scores
tree_cols = ["Horizon_Days", "Model", *METRIC_COLS]  # * unpacks METRIC_COLS into this list


def row_from_predictions(pred_csv, model_name, pred_col):
    """Score Actual_PM25 vs `pred_col` for a 1-day prediction CSV."""
    df = pd.read_csv(pred_csv)
    return pd.DataFrame([{
        "Horizon_Days": 1,
        "Model": model_name,
        **regression_metrics(df["Actual_PM25"], df[pred_col]),  # ** put MAE, RMSE etc into a separate column for each metric
    }])


def pooled_plus_lead1(raw, pred_csv, model_name):
    """7 / 14 / 30 from the pooled-path CSV, plus the 1-day specialist."""
    out = raw[tree_cols].copy()
    lead1 = row_from_predictions(pred_csv, model_name, "Predicted_PM25")
    return pd.concat([lead1, out], ignore_index=True)


rf_cmp = pooled_plus_lead1(rf, RF_PRED / "rf_predictions_1d.csv", "Random Forest")
xgb_cmp = pooled_plus_lead1(xgb, XGB_PRED / "xgb_predictions_1d.csv", "XGBoost")

# Persistence: PM2.5 at origin t, held flat over the path (same origins as the trees).
PERS_MAP = {
    "MAE": "Persistence_MAE",
    "RMSE": "Persistence_RMSE",
    "R2": "Persistence_R2",
    "MAPE": "Persistence_MAPE",
    "MSE": "Persistence_MSE",
    "MBE": "Persistence_MBE",
}
pers_rows = []
for _, row in rf.iterrows(): # for each of the horizons in the Random Forest scores, create a corresponding persistence record
    rec = {
        "Horizon_Days": int(row["Horizon_Days"]),
        "Model": "Persistence",
    }
    for dest, src in PERS_MAP.items(): #copy persistence metrics from the RF row to the new record
        rec[dest] = row[src]
    pers_rows.append(rec) # add the new persistence record to the list of persistence rows

pers_1 = row_from_predictions( # Get 1-day persistence predictions
    RF_PRED / "rf_predictions_1d.csv",
    "Persistence",
    "Persistence_PM25",
)
pers_cmp = pd.concat([pers_1, pd.DataFrame(pers_rows)], ignore_index=True) # Combine 1-day persistence with the rest of the persistence scores


## Comparison table


In [12]:
comparison_df = pd.concat(  # Combine all model comparison DataFrames into a single DataFrame
    [cnn, transformer, lstm_cmp, gru_cmp, rf_cmp, xgb_cmp, pers_cmp],
    ignore_index=True,
)
comparison_df["Evaluation"] = np.where(  # Mark 1-day predictions as 'lead_1' and others as 'pooled_path'
    comparison_df["Horizon_Days"] == 1, "lead_1", "pooled_path"
)
comparison_df = comparison_df[ # Select and order columns for comparison
    ["Model", "Horizon_Days", "Evaluation", "MAE", "RMSE", "MSE", "R2", "MAPE", "MBE"]
].sort_values(["Horizon_Days", "MAE"]).reset_index(drop=True) # Sort by prediction horizon and MAE for easier comparison

display(comparison_df.round(4))

out_path = HERE / "pooled_path_comparison.csv"
to_save = comparison_df
if out_path.exists():
    prev = pd.read_csv(out_path)
    clim = prev.loc[prev["Model"] == "Climatology", comparison_df.columns]
    if not clim.empty:
        # climatology_baseline.ipynb appends these, keep them if already present
        to_save = pd.concat([comparison_df, clim], ignore_index=True)
        to_save = to_save.drop_duplicates(["Model", "Horizon_Days"], keep="first")
        to_save = to_save.sort_values(["Horizon_Days", "MAE"]).reset_index(drop=True)
to_save.to_csv(out_path, index=False) #take the final comparison DataFrame and save it to CSV
print("Saved", out_path)


,Model,Horizon_Days,Evaluation,MAE,RMSE,MSE,R2,MAPE,MBE
0,Persistence,1,lead_1,3.1241,5.6737,32.1905,0.3742,37.6353,0.0003
1,Random Forest,1,lead_1,3.1536,5.1020,26.0302,0.4939,41.3530,0.0642
2,XGBoost,1,lead_1,3.1553,5.1732,26.7617,0.4797,40.8981,-0.0220
3,CNN-LSTM,1,lead_1,3.1659,4.8798,23.8128,0.4750,42.1094,0.1237
4,Transformer,1,lead_1,4.3642,6.7731,45.8750,-0.0113,56.9371,-0.1775
5,CNN-LSTM,7,pooled_path,3.8269,6.3926,40.8656,0.1063,45.0226,-1.0630
6,XGBoost,7,pooled_path,4.0296,6.3467,40.2808,0.2244,49.8353,-0.4194
7,Random Forest,7,pooled_path,4.0619,6.2702,39.3148,0.2430,51.3572,-0.2381
8,GRU,7,pooled_path,4.1361,6.5209,42.5225,0.1799,48.0664,-0.9694
9,LSTM,7,pooled_path,4.3211,7.0847,50.1934,0.0320,47.4427,-1.3913


Saved pooled_path_comparison.csv


## Presentable table

Rounded for the paper. Skill vs persistence: `(pers − model) / pers × 100`. Positive = better than persistence.


In [13]:
# Skill vs persistence: (pers error − model error) / pers error × 100.
# Positive = better than persistence (lower error). Persistence itself is 0.
pers = (
    comparison_df.loc[comparison_df["Model"] == "Persistence", ["Horizon_Days", "MAE", "RMSE"]]
    .rename(columns={"MAE": "Persistence_MAE", "RMSE": "Persistence_RMSE"})
)
presentable = comparison_df.merge(pers, on="Horizon_Days", how="left")
presentable["MAE_vs_persistence_pct"] = ( #percentage improvement over persistence on MAE
    (presentable["Persistence_MAE"] - presentable["MAE"]) / presentable["Persistence_MAE"] * 100
)
presentable["RMSE_vs_persistence_pct"] = ( #percentage improvement over persistence on RMSE
    (presentable["Persistence_RMSE"] - presentable["RMSE"]) / presentable["Persistence_RMSE"] * 100
)

keep = [
    "Model", "Horizon_Days", "Evaluation",
    "MAE", "RMSE", "MSE", "R2", "MAPE", "MBE",
    "MAE_vs_persistence_pct", "RMSE_vs_persistence_pct",
]
rounded = presentable[keep].copy()
decimals = { # number of decimal places to round each column to
    "MAE": 3, "RMSE": 3, "MSE": 3, "R2": 3, "MAPE": 1, "MBE": 3,
    "MAE_vs_persistence_pct": 1, "RMSE_vs_persistence_pct": 1,
}
for col, n in decimals.items(): # round each column to the specified number of decimal places
    rounded[col] = rounded[col].round(n)
rounded = rounded.rename(columns={
    "Horizon_Days": "Horizon (days)",
    "MAPE": "MAPE (%)",
    "MAE_vs_persistence_pct": "MAE vs persistence (%)",
    "RMSE_vs_persistence_pct": "RMSE vs persistence (%)",
})

display(rounded)

presentable_path = HERE / "pooled_path_comparison_presentable.csv"
rounded_out = rounded
if presentable_path.exists():
    prev = pd.read_csv(presentable_path)
    clim = prev.loc[prev["Model"] == "Climatology"]
    if not clim.empty: # if there are any Climatology rows in the previous CSV, include them
        rounded_out = pd.concat([rounded, clim], ignore_index=True)
        rounded_out = rounded_out.drop_duplicates(["Model", "Horizon (days)"], keep="first")
        rounded_out = rounded_out.sort_values(["Horizon (days)", "MAE"]).reset_index(drop=True)
rounded_out.to_csv(presentable_path, index=False)
print("Saved", presentable_path)


,Model,Horizon (days),Evaluation,MAE,RMSE,MSE,R2,MAPE (%),MBE,MAE vs persistence (%),RMSE vs persistence (%)
0,Persistence,1,lead_1,3.124,5.674,32.191,0.374,37.6,0.000,0.0,0.0
1,Random Forest,1,lead_1,3.154,5.102,26.030,0.494,41.4,0.064,-0.9,10.1
2,XGBoost,1,lead_1,3.155,5.173,26.762,0.480,40.9,-0.022,-1.0,8.8
3,CNN-LSTM,1,lead_1,3.166,4.880,23.813,0.475,42.1,0.124,-1.3,14.0
4,Transformer,1,lead_1,4.364,6.773,45.875,-0.011,56.9,-0.178,-39.7,-19.4
5,CNN-LSTM,7,pooled_path,3.827,6.393,40.866,0.106,45.0,-1.063,21.6,20.1
6,XGBoost,7,pooled_path,4.030,6.347,40.281,0.224,49.8,-0.419,17.5,20.7
7,Random Forest,7,pooled_path,4.062,6.270,39.315,0.243,51.4,-0.238,16.8,21.6
8,GRU,7,pooled_path,4.136,6.521,42.523,0.180,48.1,-0.969,15.3,18.5
9,LSTM,7,pooled_path,4.321,7.085,50.193,0.032,47.4,-1.391,11.5,11.5


Saved pooled_path_comparison_presentable.csv


## Feature importance ranks


In [14]:
IMP_HORIZONS = [1, 7, 14, 30]
IMP_MODELS = ["Random Forest", "XGBoost", "CNN-LSTM", "Transformer"]
RANK_COL = {
    "Random Forest": "rf_rank",
    "XGBoost": "xgb_rank",
    "CNN-LSTM": "cnn_lstm_rank",
    "Transformer": "transformer_rank",
}


def rank_importance(series):
    """1 = largest permutation MAE increase (most important). Ties share the min rank."""
    return series.rank(ascending=False, method="min") #method = min if two features have the same importance, they get the same rank


def importance_for(model, h):
    """Permutation (or XGB gain) scores for one model and horizon, or None if the file is missing."""
    if model == "Random Forest":
        path = HERE / "rf_results_path" / f"rf_permutation_importance_{h}d.csv"
        if not path.exists():
            return None
        return pd.read_csv(path).set_index("Feature")["Importance_Mean"].astype(float)
    if model == "XGBoost":
        path = HERE / "xgb_results_path" / "xgboost_feature_importance_across_horizons.csv"
        if not path.exists():
            return None
        df = pd.read_csv(path)
        col = f"{h} Day"
        if col not in df.columns:
            return None
        return df.set_index(df.columns[0])[col].astype(float)
    if model == "CNN-LSTM":
        path = HERE / "cnn_lstm_results" / f"cnn_lstm_permutation_importance_{h}d.csv"
        if not path.exists():
            return None
        return pd.read_csv(path).set_index("Feature")["Importance_Mean"].astype(float)
    if model == "Transformer":
        path = HERE / "transformer_results" / f"transformer_permutation_importance_{h}d.csv"
        if not path.exists():
            return None
        return pd.read_csv(path).set_index("Feature")["Importance_Mean"].astype(float)
    return None


rows = []
for h in IMP_HORIZONS:
    ranks = {}
    for model in IMP_MODELS:
        series = importance_for(model, h) #get the importance scores for current model and horizon
        if series is None:
            continue
        ranks[model] = rank_importance(series) #rank the importance scores for the current model and horizon
    if not ranks:
        continue
    rank_df = pd.concat(ranks, axis=1) #combine the ranked importance scores for all models into a single DataFrame for the current horizon
    present = list(ranks)
    out = pd.DataFrame({"horizon_days": h, "feature": rank_df.index}) #rank_df.index is all features for the current horizon (so create a row for each feature for this horizon)
    out["mean_rank"] = rank_df[present].mean(axis=1).to_numpy() # compute the mean rank across all present models for each feature, lower rank = higher average importance
    out["n_models"] = rank_df[present].notna().sum(axis=1).to_numpy() # count the number of models that have a non-NA rank for each feature
    for model in IMP_MODELS:
        out[RANK_COL[model]] = rank_df[model].to_numpy() if model in rank_df else np.nan # add the rank for the current model, or NaN if the model is not present
    rows.append(out) #append the feature importance ranks for the current horizon to the list of all rows

imp_ranks = pd.concat(rows, ignore_index=True) # combine all the feature importance ranks for all horizons into a single DataFrame
imp_ranks = imp_ranks.sort_values(["horizon_days", "mean_rank", "feature"]).reset_index(drop=True) #sort by horizon, then by mean rank (lower is more important), then by feature name (as a tiebreaker)

header = "# Rank within each model (1 = most important), then mean rank.\n"
with RANK_CSV.open("w") as f:
    f.write(header)
    imp_ranks.to_csv(f, index=False)
print("Saved", RANK_CSV)
display(imp_ranks.head())


Saved pooled_path_feature_importance_ranks.csv


,horizon_days,feature,mean_rank,n_models,rf_rank,xgb_rank,cnn_lstm_rank,transformer_rank
0,1,pm25_rollmean3,1.0,2,1.0,1.0,NaN,NaN
1,1,mean_wind_speed_knots,2.0,2,2.0,2.0,NaN,NaN
2,1,wind_speed_max_knots,3.0,2,3.0,3.0,NaN,NaN
3,1,temperature_max_c,4.0,2,4.0,4.0,NaN,NaN
4,1,temperature_avg_c,6.0,2,6.0,6.0,NaN,NaN
